# Reconnaissance d'entités nommées avec SpaCy

**TP2 – année choisie : 1960**

Ce notebook cherche les personnes, les lieux et les organisations les plus cités dans Le Soir en 1960. Comme demandé, nous utilisons le corpus en un seul fichier : `data/tmp/1960.txt`. Ce fichier a été créé dans `s2_wordcloud.ipynb` et il contient les 100 pages de l'année.

Par rapport au notebook du cours, nous avons changé trois choses :
- le texte analysé : toute l'année 1960 au lieu du premier million de caractères de `all.txt` ;
- le découpage du texte en morceaux, car SpaCy refuse les textes trop longs ;
- l'ajout des lieux (LOC) et des organisations (ORG), avec un nettoyage des résultats.

La documentation est accessible ici: https://spacy.io/api

## Imports

In [1]:
from collections import defaultdict
import spacy
from spacy.lang.fr.examples import sentences

In [2]:
nlp = spacy.load('fr_core_news_md')

## Exemple sur un corpus de test fourni par SpaCy

In [3]:
# Imprimer le corpus de Spacy
sentences

['Apple cherche à acheter une start-up anglaise pour 1 milliard de dollars',
 "Les voitures autonomes déplacent la responsabilité de l'assurance vers les constructeurs",
 "San Francisco envisage d'interdire les robots coursiers sur les trottoirs",
 'Londres est une grande ville du Royaume-Uni',
 'L’Italie choisit ArcelorMittal pour reprendre la plus grande aciérie d’Europe',
 "Apple lance HomePod parce qu'il se sent menacé par l'Echo d'Amazon",
 "La France ne devrait pas manquer d'électricité cet été, même en cas de canicule",
 'Nouvelles attaques de Trump contre le maire de Londres',
 'Où es-tu ?',
 'Qui est le président de la France ?',
 'Où est la capitale des États-Unis ?',
 'Quand est né Barack Obama ?']

In [4]:
# Isoler la première phrase
sent = sentences[0]
sent

'Apple cherche à acheter une start-up anglaise pour 1 milliard de dollars'

In [5]:
# Traiter la phrase avec Spacy
doc = nlp(sent)

In [6]:
type(doc)

spacy.tokens.doc.Doc

In [7]:
doc.text

'Apple cherche à acheter une start-up anglaise pour 1 milliard de dollars'

In [8]:
# Appliquer le test sur toutes les phrases
for sent in sentences:
    doc = nlp(sent)
    entities = []
    for ent in doc.ents:
        entities.append(f"{ent.text} ({ent.label_})")
    if entities:
        print(f"'{doc.text}' contient les entités suivantes : {', '.join(entities)}")
    else:
        print(f"'{doc.text}' ne contient aucune entité")

'Apple cherche à acheter une start-up anglaise pour 1 milliard de dollars' contient les entités suivantes : Apple (ORG)
'Les voitures autonomes déplacent la responsabilité de l'assurance vers les constructeurs' ne contient aucune entité
'San Francisco envisage d'interdire les robots coursiers sur les trottoirs' contient les entités suivantes : San Francisco (LOC)
'Londres est une grande ville du Royaume-Uni' contient les entités suivantes : Londres (LOC), Royaume-Uni (LOC)
'L’Italie choisit ArcelorMittal pour reprendre la plus grande aciérie d’Europe' contient les entités suivantes : L’Italie (LOC), ArcelorMittal (ORG), Europe (LOC)
'Apple lance HomePod parce qu'il se sent menacé par l'Echo d'Amazon' contient les entités suivantes : Apple (ORG), HomePod (MISC), Echo (ORG), Amazon (ORG)
'La France ne devrait pas manquer d'électricité cet été, même en cas de canicule' contient les entités suivantes : La France (LOC)
'Nouvelles attaques de Trump contre le maire de Londres' contient les en

## Appliquer la reconnaissance d'entités nommées sur notre corpus

In [9]:
# Choisir une année
year = 1960

In [10]:
# Charger le texte
# On utilise le corpus de l'année en un seul fichier (créé dans s2_wordcloud.ipynb)
text = open(f"../../data/tmp/{year}.txt", encoding='utf-8').read()
len(text)

3338805

Le texte de 1960 contient 3 338 805 caractères. Or SpaCy accepte au maximum 1 000 000 de caractères à la fois. Nous découpons donc le texte en 34 morceaux. Nous coupons toujours sur un espace, pour ne pas couper un mot ou un nom en deux.

In [11]:
# SpaCy refuse les textes de plus de 1 000 000 de caractères.
# On découpe donc le texte en morceaux de 100 000 caractères environ, en coupant sur un espace.
size = 100000
chunks = []
start = 0
while start < len(text):
    end = min(start + size, len(text))
    if end < len(text):
        space = text.rfind(" ", start, end)
        if space > start:
            end = space
    chunks.append(text[start:end])
    start = end
len(chunks)

34

In [12]:
%%time
# Traiter le texte

docs = list(nlp.pipe(chunks))

CPU times: total: 2min 45s
Wall time: 2min 55s


### Premier essai : compter les entités sans nettoyage

In [13]:
# Compter les entités : personnes (PER), lieux (LOC) et organisations (ORG)
people = defaultdict(int)
locations = defaultdict(int)
organisations = defaultdict(int)
for doc in docs:
    for ent in doc.ents:
        name = " ".join(ent.text.split())
        if len(name) > 3:
            if ent.label_ == "PER":
                people[name] += 1
            elif ent.label_ == "LOC":
                locations[name] += 1
            elif ent.label_ == "ORG":
                organisations[name] += 1

print(f"{len(people)} personnes, {len(locations)} lieux et {len(organisations)} organisations différentes")
print(f"{sum(people.values())} mentions de personnes, {sum(locations.values())} mentions de lieux et {sum(organisations.values())} mentions d'organisations")

16050 personnes, 12863 lieux et 4841 organisations différentes
21091 mentions de personnes, 21552 mentions de lieux et 6672 mentions d'organisations


In [14]:
# Trier et imprimer les personnes (comme dans le notebook du cours)

sorted_people = sorted(people.items(), key=lambda kv: kv[1], reverse=True)

for person, freq in sorted_people[:30]:
    print(f"{person} apparait {freq} fois dans le corpus")

Rossel apparait 424 fois dans le corpus
culs apparait 76 fois dans le corpus
Ecrire Ag apparait 55 fois dans le corpus
Louise apparait 54 fois dans le corpus
Van Looy apparait 40 fois dans le corpus
Albert apparait 37 fois dans le corpus
Marie apparait 36 fois dans le corpus
curr apparait 35 fois dans le corpus
trav apparait 34 fois dans le corpus
- Tél apparait 32 fois dans le corpus
Agence Rossel apparait 30 fois dans le corpus
Jean apparait 29 fois dans le corpus
Bonne apparait 27 fois dans le corpus
O. N. U. apparait 27 fois dans le corpus
Reine apparait 22 fois dans le corpus
Robert apparait 22 fois dans le corpus
Jard apparait 22 fois dans le corpus
M. Khrouchtchev apparait 22 fois dans le corpus
général de Gaulle apparait 21 fois dans le corpus
curric apparait 20 fois dans le corpus
Roger apparait 20 fois dans le corpus
Janssens apparait 19 fois dans le corpus
Monsieur apparait 19 fois dans le corpus
A. P. apparait 19 fois dans le corpus
Molière apparait 18 fois dans le corpus
E

**Commentaire.** SpaCy trouve 21 091 mentions de personnes, 21 552 mentions de lieux et 6 672 mentions d'organisations. Cependant, le classement des personnes est décevant. La première « personne » est « Rossel » avec 424 mentions. Or ce nom vient de l'Agence Rossel, citée dans les petites annonces. On trouve aussi « culs » (76 fois), « Ecrire Ag » ou « trav », qui sont des abréviations d'annonces. Viennent ensuite des prénoms seuls (« Louise », « Albert », « Marie »), qui sont souvent des noms de rues, comme l'avenue Louise.

SpaCy se trompe pour deux raisons. D'abord, le modèle a été entraîné sur des textes modernes et propres, alors que notre texte est ancien et abîmé par l'OCR. Ensuite, les petites annonces sont écrites en style télégraphique, sans vraies phrases. Le modèle manque donc de contexte. Il faut nettoyer les résultats.

### Nettoyer les entités

Nous nettoyons les entités en quatre étapes :
1. nous retirons le titre au début du nom. Ainsi, « M. Lumumba » et « Lumumba » sont comptés ensemble ;
2. nous regroupons les sigles et les majuscules. Ainsi, « O. N. U. » et « O.N.U. » sont comptés ensemble, tout comme « BRUXELLES » et « Bruxelles » ;
3. nous écartons les noms sans majuscule au début, car ce sont presque toujours des abréviations d'annonces ;
4. nous écartons une liste de 145 fausses entités. Nous avons construit cette liste de manière itérative, comme la liste de stopwords : nous avons regardé le classement, noté les erreurs, puis recommencé.

In [15]:
# Liste des fausses entités, construite de manière itérative (comme la liste de stopwords)
noise = ["rossel", "agence rossel", "agence rossel num", "rossel num", "ag rossel", "rossol", "ecrire ag", "ecag", "ec ag",
         "brux", "téléph", "téléphone", "trav", "jard", "créd", "holl", "inut", "rens",
         "appart", "dlst", "inst", "référ", "import", "inter", "cand", "s'adr", "on dem",
         "k dem", "s.a", "a.p", "j.-p", "g.-b", "e.-u", "f.c", "t.c.l", "mutuel",
         "jumelé", "monsieur", "dame", "reine", "bonne", "gros", "fils", "lundi", "midi",
         "nord", "centre", "jardin", "bourse", "etat", "cycliste", "louise", "albert", "marie",
         "jean", "robert", "roger", "joseph", "louis", "jacques", "paul", "marcel", "henry",
         "henri", "michel", "gérard", "charles", "jules", "laurent", "hugues", "arnold", "judith",
         "elisabeth", "léopold", "piétro", "christ", "belge", "belges", "français", "congolais", "européens",
         "noirs", "hollandais", "bruxellois", "américains", "allemands", "anglais", "soviétiques", "blancs", "conseil",
         "cour", "firme", "employee", "credit", "parc", "galeries", "assurances", "manœuvres", "promotion",
         "international", "achete", "intimité", "criterium", "usines", "publicité", "facturiere", "alimentât", "demandons",
         "emballeur", "dactylo", "repasseuses", "diesel", "mecaniciens", "garage", "gaite", "travail", "couronne",
         "post", "representant", "representants", "nations", "quatrième course", "union", "zuen", "comité", "garantie",
         "taxis", "football", "fauteuils", "central", "alcazar", "royal", "libre", "faire", "fixe",
         "ecrire agence rossel", "mardi", "jeudi", "ouvr", "ameri", "société", "hotel", "techn", "bois",
         "leopold"]
noise = set(noise)
len(noise)

145

In [16]:
import re

titles = ("M. ", "MM. ", "Mme ", "Mlle ", "Mgr ", "le ", "la ", "les ", "A ", "l'", "l’", "du ", "de ", "au ", "en ")

def clean_entity(name):
    """Nettoie le nom d'une entité. Renvoie None si l'entité doit être écartée."""
    # Retirer le titre ou l'article au début ("M. Lumumba" devient "Lumumba")
    for t in titles:
        if name.startswith(t):
            name = name[len(t):]
    name = name.strip(" .,-:;'’\"«»")
    # Regrouper les sigles écrits avec des espaces ("O. N. U." devient "O.N.U")
    if re.fullmatch(r"([A-Z]\. ?)+[A-Z]?", name):
        name = name.replace(" ", "")
    # Regrouper les majuscules et les minuscules ("BRUXELLES" devient "Bruxelles")
    if name.isupper() and "." not in name:
        name = name.title()
    # Ecarter les noms trop courts, sans majuscule au début ou avec un chiffre
    if len(name) <= 3 or not name[0].isupper() or any(c.isdigit() for c in name):
        return None
    # Ecarter les fausses entités de la liste
    if name.lower() in noise:
        return None
    return name

clean_entity("M. Lumumba"), clean_entity("O. N. U."), clean_entity("BRUXELLES"), clean_entity("trav"), clean_entity("Rossel")

('Lumumba', 'O.N.U', 'Bruxelles', None, None)

In [17]:
# Compter à nouveau les entités après nettoyage
people = defaultdict(int)
locations = defaultdict(int)
organisations = defaultdict(int)
for doc in docs:
    for ent in doc.ents:
        name = clean_entity(" ".join(ent.text.split()))
        if name:
            if ent.label_ == "PER":
                people[name] += 1
            elif ent.label_ == "LOC":
                locations[name] += 1
            elif ent.label_ == "ORG":
                organisations[name] += 1

print(f"{len(people)} personnes, {len(locations)} lieux et {len(organisations)} organisations différentes")
print(f"{sum(people.values())} mentions de personnes, {sum(locations.values())} mentions de lieux et {sum(organisations.values())} mentions d'organisations")

12296 personnes, 8959 lieux et 3758 organisations différentes
15740 mentions de personnes, 15578 mentions de lieux et 4877 mentions d'organisations


**Commentaire.** Après nettoyage, il reste 15 740 mentions de personnes (au lieu de 21 091), 15 578 mentions de lieux (au lieu de 21 552) et 4 877 mentions d'organisations (au lieu de 6 672). Nous avons donc écarté environ un quart des mentions dans chaque catégorie (25 % des personnes, 28 % des lieux et 27 % des organisations). Ce chiffre montre que le bruit est important dans ce corpus.

### Les personnes (PER)

In [18]:
sorted_people = sorted(people.items(), key=lambda kv: kv[1], reverse=True)

for person, freq in sorted_people[:40]:
    print(f"{person} apparait {freq} fois dans le corpus")

Van Looy apparait 40 fois dans le corpus
A.F.P apparait 33 fois dans le corpus
O.N.U apparait 27 fois dans le corpus
Janssens apparait 26 fois dans le corpus
Khrouchtchev apparait 26 fois dans le corpus
Lumumba apparait 23 fois dans le corpus
Tshombe apparait 20 fois dans le corpus
Molière apparait 18 fois dans le corpus
Brugmann apparait 18 fois dans le corpus
Dumont apparait 17 fois dans le corpus
Gilson apparait 16 fois dans le corpus
Daems apparait 16 fois dans le corpus
Desmet apparait 16 fois dans le corpus
Elisabethville apparait 16 fois dans le corpus
Mertens apparait 14 fois dans le corpus
Hugo apparait 14 fois dans le corpus
Eyskens apparait 13 fois dans le corpus
Eisenhower apparait 13 fois dans le corpus
Kennedy apparait 13 fois dans le corpus
Verbiest apparait 13 fois dans le corpus
Mairesse apparait 13 fois dans le corpus
Jette apparait 12 fois dans le corpus
Lejeune apparait 12 fois dans le corpus
Chopin apparait 12 fois dans le corpus
Gilbert Desmet apparait 11 fois dan

**Commentaire (personnes).** Le classement a maintenant du sens. Nous y voyons quatre groupes.

- **Les acteurs de la crise congolaise** : Lumumba (23 mentions), Tshombe (20) et Hammarskjöld (10), le secrétaire général de l'ONU. Lumumba est Premier ministre du Congo et Tshombe dirige le Katanga, qui fait sécession. Le Premier ministre belge Eyskens (13) appartient aussi à ce groupe.
- **Les dirigeants de la guerre froide** : Khrouchtchev (26), Eisenhower (13) et Kennedy (13). Khrouchtchev est donc plus cité que Lumumba. Cela montre que le journal suit de près les relations entre l'URSS et les États-Unis. Kennedy est élu président en novembre 1960.
- **Les sportifs** : Van Looy est la personne la plus citée de l'année avec 40 mentions, et même 49 si on ajoute « Rik Van Looy ». Daems (16) et Desmet (16) sont aussi des coureurs cyclistes. Le sport occupe donc une grande place dans le journal.
- **Les artistes** : Molière (18), Hugo (14), Chopin (12) et Bach (10) viennent des pages culturelles et des programmes de radio.

**Limites.** Il reste des erreurs. « A.F.P », « O.N.U » et « P.S.C » sont des organisations, mais SpaCy les classe comme des personnes. « Brugmann », « Jette » et « Elisabethville » sont des lieux. De plus, certains noms de famille comme « Janssens » ou « Dumont » sont très courants en Belgique. Ils désignent donc sûrement plusieurs personnes différentes. Enfin, les chiffres restent faibles : 23 mentions pour Lumumba sur 100 pages. Cela s'explique par l'échantillon, qui contient beaucoup de pages d'annonces et de sport.

### Les lieux (LOC)

In [19]:
sorted_locations = sorted(locations.items(), key=lambda kv: kv[1], reverse=True)

for location, freq in sorted_locations[:40]:
    print(f"{location} apparait {freq} fois dans le corpus")

Bruxelles apparait 617 fois dans le corpus
Belgique apparait 211 fois dans le corpus
Congo apparait 181 fois dans le corpus
Anvers apparait 140 fois dans le corpus
France apparait 112 fois dans le corpus
Paris apparait 112 fois dans le corpus
Liège apparait 96 fois dans le corpus
Gand apparait 69 fois dans le corpus
Mons apparait 62 fois dans le corpus
Anderlecht apparait 61 fois dans le corpus
Londres apparait 58 fois dans le corpus
Namur apparait 57 fois dans le corpus
Etats-Unis apparait 56 fois dans le corpus
Uccle apparait 52 fois dans le corpus
Ixelles apparait 49 fois dans le corpus
Katanga apparait 49 fois dans le corpus
Louvain apparait 48 fois dans le corpus
Schaerbeek apparait 43 fois dans le corpus
Léopoldville apparait 42 fois dans le corpus
Europe apparait 41 fois dans le corpus
Wavre apparait 38 fois dans le corpus
Allemagne apparait 38 fois dans le corpus
Italie apparait 35 fois dans le corpus
Luxembourg apparait 34 fois dans le corpus
Forest apparait 34 fois dans le co

**Commentaire (lieux).** Les lieux sont mieux reconnus que les personnes. Trois niveaux apparaissent.

- **Bruxelles domine très largement** avec 617 mentions, soit presque trois fois plus que la Belgique (211). Ce résultat est normal, car Le Soir est un journal bruxellois. De plus, beaucoup de ces mentions viennent des adresses des annonces. Les communes bruxelloises confirment cette explication : Anderlecht (61), Uccle (52), Ixelles (49) et Schaerbeek (43).
- **Le Congo est le premier lieu étranger** avec 181 mentions. Si on ajoute le Katanga (49) et Léopoldville (42), on arrive à 272 mentions. Le Congo dépasse donc la France (112) et Paris (112). C'est la marque de l'année 1960.
- **Les lieux de la guerre froide** sont aussi présents : États-Unis (56), New York (33) et Moscou (32). New York est le siège de l'ONU, où la crise congolaise est discutée.

### Les organisations (ORG)

In [20]:
sorted_organisations = sorted(organisations.items(), key=lambda kv: kv[1], reverse=True)

for organisation, freq in sorted_organisations[:40]:
    print(f"{organisation} apparait {freq} fois dans le corpus")

Fiat apparait 28 fois dans le corpus
Opel apparait 25 fois dans le corpus
Standard apparait 22 fois dans le corpus
Parlement apparait 22 fois dans le corpus
Benelux apparait 21 fois dans le corpus
Chambre apparait 21 fois dans le corpus
O.N.U apparait 21 fois dans le corpus
Racing apparait 17 fois dans le corpus
Peugeot apparait 15 fois dans le corpus
Evere apparait 13 fois dans le corpus
Reuter apparait 13 fois dans le corpus
Nations Unies apparait 13 fois dans le corpus
Académie apparait 12 fois dans le corpus
Haute Autorité apparait 11 fois dans le corpus
Ford apparait 11 fois dans le corpus
Vilvorde apparait 9 fois dans le corpus
Philips apparait 9 fois dans le corpus
Anderlecht apparait 8 fois dans le corpus
Michelin apparait 8 fois dans le corpus
Congo belge apparait 7 fois dans le corpus
Université apparait 7 fois dans le corpus
Justice apparait 7 fois dans le corpus
Sénat apparait 7 fois dans le corpus
Classes moyennes apparait 7 fois dans le corpus
General Motors apparait 7 fo

**Commentaire (organisations).** C'est la catégorie la plus difficile pour SpaCy. Elle compte trois fois moins de mentions que les deux autres. Trois types d'organisations ressortent.

- **Les marques de voitures** : Fiat (28), Opel (25), Peugeot (15) et Ford (11). Elles viennent des publicités et des annonces de voitures d'occasion. Elles montrent la place de l'automobile dans la société de 1960.
- **Les institutions politiques** : le Parlement (22), la Chambre (21), le Benelux (21) et le Sénat (7). L'ONU apparaît sous deux formes, « O.N.U » (21) et « Nations Unies » (13), auxquelles s'ajoute le Conseil de sécurité (7). Cela fait 41 mentions au total. L'ONU est donc l'organisation politique la plus citée, ce qui confirme son rôle dans la crise congolaise.
- **Les clubs de football** : le Standard (22) et le Racing (17).

**Limites.** Certaines communes sont classées comme organisations (Evere, Vilvorde, Anderlecht). Dans le cas d'Anderlecht, il peut s'agir du club de football. SpaCy ne peut pas toujours faire la différence.

**Conclusion.** Les trois catégories racontent la même histoire. L'actualité de 1960 dans Le Soir tourne autour du Congo (Lumumba, Tshombe, Katanga, Léopoldville, ONU) et de la guerre froide (Khrouchtchev, Eisenhower, Moscou). À côté de l'actualité, le journal donne une grande place au sport, à la vie bruxelloise et à la publicité.